# 05 - Learned Embeddings: Word2Vec (CBOW, Skip-gram) & fastText (Parts B, C)
**Owner:** Omais Rizvi

Models are trained once by `src/train_embeddings.py` and saved in `models/`. Here we only LOAD them.

In [8]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/cleaned.csv").dropna(subset=["clean_text"])
docs, labels = df["clean_text"].tolist(), df["label"].tolist()
print(len(docs), "documents")

40000 documents


In [10]:
from gensim.models import Word2Vec, FastText
from train_embeddings import PARAMS
MODELS = ROOT / "models"
cbow = Word2Vec.load(str(MODELS / "w2v_cbow.model")).wv
skip = Word2Vec.load(str(MODELS / "w2v_skipgram.model")).wv
ft   = FastText.load(str(MODELS / "fasttext.model")).wv
models = {"CBOW": cbow, "Skip-gram": skip, "fastText": ft}
print("Hyperparameters (same for all 3):", PARAMS)
for n, m in models.items(): print(f"{n:10s} vocab={len(m):,}  dim={m.vector_size}")

Hyperparameters (same for all 3): {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5, 'workers': 4, 'seed': 42}
CBOW       vocab=29,304  dim=100
Skip-gram  vocab=29,304  dim=100
fastText   vocab=29,304  dim=100


## B1/B2. Hyperparameters
| Parameter | Value | Why |
|---|---|---|
| vector_size | 100 | enough for a ~40k-article corpus, small to train |
| window | 5 | 5 words on each side as context |
| min_count | 5 | ignore very rare words (mostly typos / glued words) |
| epochs | 5 | gensim default |
| sg | 0 = CBOW, 1 = Skip-gram | the ONLY difference between B1 and B2 |
| fastText | sg=1, char n-grams 3-6, bucket=200,000 | same settings + sub-words (bucket keeps the saved file small) |

## B3. CBOW vs Skip-gram (edit in your own words)
- **CBOW** looks at the context words around a gap and predicts the **center** word (context -> center). It averages the context, so it is faster and smooth for frequent words.
- **Skip-gram** takes the **center** word and predicts each context word (center -> context). Every word gets more training updates, so it learns **rare words** better, but it is slower.
- Our expectation: Skip-gram should do better on our data, because Urdu news has many rare names and spellings, and our corpus is medium-sized.
- **What we saw:** Skip-gram was better for موبائل and gave higher similarity scores for related pairs (0.716 vs 0.542 for بینک / قرض). CBOW was cleaner for فلم and معیشت and was much faster to train (21.7s vs 96.9s for Skip-gram and 170.1s for fastText).

## C1. Most similar words

In [11]:
queries = ["پاکستان", "کرکٹ", "فلم", "معیشت", "موبائل"]   # change to words meaningful for you
for q in queries:
    print(f"\n=== {q} ===")
    for n, m in models.items():
        if q in m.key_to_index:
            print(f"{n:10s}", [w for w, s in m.most_similar(q, topn=6)])
        else:
            print(f"{n:10s} (not in vocabulary)")


=== پاکستان ===
CBOW       ['پاکستانی', 'ملک', 'قومی', 'بھارت', 'افغانستان', 'ہماری']
Skip-gram  ['پاکستا', 'ملک', 'اورپاکستان', 'انویسٹمینٹ', 'اڈی', 'کراچین']
fastText   ['توپاکستان', 'کہپاکستان', 'پاکستانکے', 'کوپاکستان', 'بعدپاکستان', 'ریڈیوپاکستان']

=== کرکٹ ===
CBOW       ['ہاکی', 'کرکٹرز', 'دورئہ', 'سپورٹس', 'کرکٹر', 'سلیکشن']
Skip-gram  ['ٹوئنٹی', 'کرکٹرز', 'اورکرکٹ', 'کرکٹر', 'ڈومیسٹک', 'ٹونٹی']
fastText   ['گاکرکٹ', 'کوکرکٹ', 'گیکرکٹ', 'کرکٹرزکو', 'تھاکرکٹ', 'کرکٹرزنے']

=== فلم ===
CBOW       ['سیکوئل', 'فلمز', 'کامیڈی', 'تھافلم', 'فلمساز', 'رومانس']
Skip-gram  ['سپرہٹ', 'رومانٹک', 'تھیفلم', 'ہوگیفلم', 'ہدایتکار', 'سیکوئل']
fastText   ['پرفلم', 'وڈفلم', 'دیفلم', 'دوفلم', 'کیفلم', 'فلمو']

=== معیشت ===
CBOW       ['معاشی', 'معشیت', 'اقتصادی', 'تشخص', 'سلامتی', 'تعمیروترقی']
Skip-gram  ['معاشی', 'معشیت', 'معیشیت', 'وخوشحالی', 'اورمعیشت', 'پالیس']
fastText   ['معیشیت', 'اورمعیشت', 'معاشی', 'ومعاشی', 'معشیت', 'وصنعت']

=== موبائل ===
CBOW       ['سمارٹ', 'اسمارٹ', 'ءموبائل', '

**Comment:** CBOW and Skip-gram give neighbours that make sense. For فلم we got سیکوئل، فلمز، کامیڈی، فلمساز, for معیشت we got معاشی، اقتصادی, and for موبائل Skip-gram gave فون، فونز، ٹیبلٹس. fastText is different: its neighbours are words like توپاکستان، کوکرکٹ، پرفلم. These are words where the word is glued to another word after cleaning, and fastText groups them because they share the same letters (character n-grams). So fastText matches by spelling/shape, while CBOW and Skip-gram match by meaning. Skip-gram also gave some odd rare words for پاکستان (like پاکستا and اڈی).

## C2. Cosine similarity: expected similar vs unrelated

In [12]:
pairs = [("کرکٹ", "میچ"), ("بینک", "قرض"), ("کرکٹ", "بجٹ"), ("فلم", "بینک")]
rows = []
for a, b in pairs:
    r = {"pair": f"{a} / {b}"}
    for n, m in models.items():
        r[n] = round(float(m.similarity(a, b)), 3) if a in m.key_to_index and b in m.key_to_index else None
    rows.append(r)
pd.DataFrame(rows)

,pair,CBOW,Skip-gram,fastText
0,کرکٹ / میچ,0.444,0.490,0.486
1,بینک / قرض,0.542,0.716,0.725
2,کرکٹ / بجٹ,-0.055,0.093,0.122
3,فلم / بینک,-0.098,0.099,0.169


**Comment:** Yes, the related pairs have a higher similarity than the unrelated ones in all three models. For بینک / قرض we got 0.542 (CBOW), 0.716 (Skip-gram) and 0.725 (fastText). For the unrelated pairs the scores are very low: کرکٹ / بجٹ is -0.055 for CBOW and فلم / بینک is -0.098. CBOW gives the lowest scores overall, while Skip-gram and fastText give higher scores even for unrelated pairs (0.09 to 0.17), but still much lower than the related pairs.

## C3. Analogy: a - b + c = ?

In [13]:
a, b, c = "لاہور", "پاکستان", "بھارت"     # Lahore - Pakistan + India = ? (expect Delhi / Mumbai)
for n, m in models.items():
    try:
        print(f"{n:10s}", m.most_similar(positive=[a, c], negative=[b], topn=5))
    except KeyError as e:
        print(f"{n:10s} missing word: {e}")

CBOW       [('ممبئی', 0.49203646183013916), ('کراچی', 0.4247313439846039), ('ملتان', 0.41295284032821655), ('لاہورسپورٹس', 0.39528730511665344), ('بھارتی', 0.38482579588890076)]
Skip-gram  [('ممبئی', 0.5769885182380676), ('امرتسر', 0.5509747266769409), ('لدھیانہ', 0.5472590327262878), ('لکھنو', 0.5394521951675415), ('لاہوراردو', 0.5386037230491638)]
fastText   [('لاہوربھارت', 0.6953046917915344), ('لاہوربھارتی', 0.6677445769309998), ('لاہورسے', 0.638676106929779), ('لاہو', 0.6314658522605896), ('لاہورلاہور', 0.6293132901191711)]


**Result:** For لاہور - پاکستان + بھارت we expected Delhi or Mumbai. CBOW gave ممبئی first (then کراچی، ملتان), and Skip-gram also gave ممبئی first (then امرتسر، لدھیانہ، لکھنؤ, which are Indian cities). So both found an Indian city, but neither gave Delhi. fastText failed: all its answers were words glued with لاہور (like لاہوربھارت), because its vectors follow spelling. The corpus is small (40k articles, about 5.6 million words), so analogies are not very reliable.

## B4. fastText and rare / misspelled words
Word2Vec has one vector per WHOLE word, so a word it never saw has no vector at all.
fastText also learns vectors for character n-grams (3-6 letters) and builds a word from its pieces, so it can make a vector for a misspelled or unseen word.

In [14]:
tests = ["پاکستانن", "کرکٹرز", "حکومتت"]     # misspelled / unseen forms
for w in tests:
    print(f"\n=== {w} ===")
    for n in ["CBOW", "Skip-gram"]:
        print(f"{n:10s}", "in vocab" if w in models[n].key_to_index else "NOT in vocabulary -> no vector")
    print("fastText  ", "in vocab" if w in ft.key_to_index else "not in vocab, but vector built from char n-grams:")
    print("           nearest:", [x for x, s in ft.most_similar(w, topn=5)])


=== پاکستانن ===
CBOW       NOT in vocabulary -> no vector
Skip-gram  NOT in vocabulary -> no vector
fastText   not in vocab, but vector built from char n-grams:
           nearest: ['پاکستانکے', 'پاکستانایس', 'پاکستانپی', 'پاکستانیو', 'پاکستانٹی']

=== کرکٹرز ===
CBOW       in vocab
Skip-gram  in vocab
fastText   in vocab
           nearest: ['کرکٹرزکو', 'کرکٹرزنے', 'کرکٹرزکی', 'کرکٹرزکے', 'کرکٹرکا']

=== حکومتت ===
CBOW       NOT in vocabulary -> no vector
Skip-gram  NOT in vocabulary -> no vector
fastText   not in vocab, but vector built from char n-grams:
           nearest: ['حکومتی', 'حکومت', 'گاحکومت', 'اورحکومت', 'پرحکومت']


**Comment:** For the misspelled words پاکستانن and حکومتت, CBOW and Skip-gram have no vector at all because they never saw these words. fastText still built a vector from character n-grams. For حکومتت its nearest words were حکومتی and حکومت, which is correct. For پاکستانن its neighbours were words glued to پاکستان (like پاکستانکے), which are related but not clean. This is the main advantage of fastText.